# Gradient Descent on Structured Quadratics

- Module: 01 Optimization
- Topic: deterministic first-order methods
- Estimated runtime: 8 minutes
- Prerequisites: gradients, Hessians, quadratic forms, Python arrays
- Expected outputs: contour plots of optimizer trajectories, convergence curves, and a learning-rate failure experiment

## Learning goals

- Connect the update rule $x_{k+1} = x_k - \eta 
abla f(x_k)$ to the geometry of a quadratic surface.
- See how curvature and step size jointly control convergence speed.
- Observe a clean failure mode when the learning rate is too large for the local smoothness scale.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().parents[2]
src_dir = repo_root / "modules" / "01-optimization" / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from optim_utils import (
    adagrad,
    adam,
    gradient_descent,
    make_quadratic,
    make_shifted_least_squares,
    momentum_descent,
    newton_method,
    plot_convergence,
    plot_trajectories,
    rmsprop,
    rosenbrock_objective,
    saddle_objective,
    stochastic_gradient_descent,
    summarize_history,
)

np.set_printoptions(precision=4, suppress=True)


## Controlled objective: a well-conditioned quadratic

We start with a symmetric positive-definite quadratic centered at the origin. Because the level sets are ellipses, the trajectory makes the geometry of gradient descent easy to interpret.

In [ ]:
objective = make_quadratic(
    matrix=np.array([[3.0, 0.8], [0.8, 1.5]], dtype=float),
    center=np.array([0.0, 0.0], dtype=float),
    name="well_conditioned_quadratic",
)
start = np.array([2.6, -2.2], dtype=float)
learning_rates = [0.15, 0.45]
histories = [gradient_descent(objective, start=start, learning_rate=rate, steps=18) for rate in learning_rates]
[summary := summarize_history(history, objective) | {"name": history["name"], "learning_rate": rate} for rate, history in zip(learning_rates, histories)]

## Visualize the trajectories

Both runs descend the same surface, but the larger stable step size takes longer moves across the contours. The path should still point roughly orthogonally to the contour lines because the gradient is normal to level sets.

In [ ]:
labeled_histories = []
for rate, history in zip(learning_rates, histories):
    labeled_history = dict(history)
    labeled_history["name"] = f"GD, lr={rate}"
    labeled_histories.append(labeled_history)

fig, _ = plot_trajectories(
    objective,
    labeled_histories,
    x_range=(-3.0, 3.0),
    y_range=(-3.0, 3.0),
    title="Gradient descent on a well-conditioned quadratic",
)
plt.show()
plt.close(fig)

## Convergence rates are visible in the loss curves

On a strongly convex quadratic, gradient descent contracts linearly when the learning rate is inside the stability interval. The empirical curves make that rate difference visible without requiring the full proof.

In [ ]:
fig, _ = plot_convergence(labeled_histories, title="Stable learning-rate comparison")
plt.show()
plt.close(fig)

## What happens when the problem is ill-conditioned?

Ill-conditioning stretches the contours into a narrow valley. Gradient descent still converges with a careful step size, but it zig-zags because the gradient points mostly across the valley rather than along it.

In [ ]:
ill_conditioned = make_quadratic(
    matrix=np.array([[20.0, 0.0], [0.0, 0.5]], dtype=float),
    center=np.array([0.0, 0.0], dtype=float),
    name="ill_conditioned_quadratic",
)
ill_history = gradient_descent(ill_conditioned, start=np.array([2.0, 2.0]), learning_rate=0.08, steps=28)
ill_history["name"] = "GD on ill-conditioned quadratic"
fig, _ = plot_trajectories(
    ill_conditioned,
    [ill_history],
    x_range=(-2.5, 2.5),
    y_range=(-0.5, 2.5),
    title="Ill-conditioning slows gradient descent",
)
plt.show()
plt.close(fig)
summarize_history(ill_history, ill_conditioned)

## Failure mode: an unstable step size

For a quadratic with largest eigenvalue $L$, the simple gradient descent update becomes unstable when the learning rate is too large relative to $1/L$. Instead of shrinking toward the minimizer, the iterates jump across the valley and the objective grows.

In [ ]:
unstable_history = gradient_descent(ill_conditioned, start=np.array([2.0, 2.0]), learning_rate=0.13, steps=12)
unstable_history["name"] = "GD, lr=0.13 (unstable)"
fig, _ = plot_trajectories(
    ill_conditioned,
    [ill_history, unstable_history],
    x_range=(-4.0, 4.0),
    y_range=(-1.0, 3.0),
    title="Stable versus unstable learning rate",
)
plt.show()
plt.close(fig)

fig, _ = plot_convergence([ill_history, unstable_history], title="Failure mode comparison")
plt.show()
plt.close(fig)

{
    "stable_final_value": summarize_history(ill_history, ill_conditioned)["final_value"],
    "unstable_final_value": summarize_history(unstable_history, ill_conditioned)["final_value"],
}

## Summary

Gradient descent is geometrically simple but highly sensitive to curvature and step size. On smooth, well-conditioned quadratics it behaves predictably; on narrow valleys or with an aggressive step size, the same update rule becomes slow or unstable.

## Next steps

- Derive the exact stability interval for a quadratic with Hessian eigenvalues $\lambda_i$.
- Change the starting point and see which visual features stay invariant.
- Compare these runs with momentum-based methods in the next notebook.